In [ ]:
import pandas as pd
import mlxtend
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

In [ ]:
df = pd.read_csv('test.csv', na_values='?')
df.drop(columns=['Unnamed: 0'], inplace=True)
df = df.dropna()
## #data preprocessing 
#convert categorical to numerical Gender,Customer Type, Type of Travel,Class
#Feature engineering

#Total delay rather than 
df['Total Delay'] = df['Departure Delay in Minutes'] + df['Arrival Delay in Minutes']

#Turns delay into a categorical feature 
def bucket_total_delay(minutes):
    if minutes <= 15:
        return 'On Time'
    elif minutes <= 60:
        return 'Small Delay'
    else:
        return 'Big Delay'

# Apply the function to create a new feature
df['Delay Category'] = df['Total Delay'].apply(bucket_total_delay)

def bucket_rating(x):
    if x <= 2:
        return 'Low'
    elif x == 3:
        return 'Medium'
    else:
        return 'High'

rating_columns = [ 'Inflight wifi service','Food and drink', 'Seat comfort',
                  'Inflight entertainment', 'On-board service', 'Leg room service',
                  'Baggage handling', 'Checkin service', 'Cleanliness', 'Online boarding']

for col in rating_columns:
    df[col] = df[col].apply(bucket_rating)

print(df.describe())
df.head(10)

In [ ]:
attr = df.drop(columns=['satisfaction', 'id'])
target = df['satisfaction']
attr.head()
categorical_columns = [
    'Gender',
    'Customer Type',
    'Type of Travel',
    'Class',
    'Inflight wifi service',
    'Food and drink',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Cleanliness',
    'Online boarding',
    'Delay Category'
]
# Create preprocessor for categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), categorical_columns) ])

attr_train, attr_test, target_train,target_test  = train_test_split(attr, target,test_size = 0.3, random_state = 5, shuffle = True)
target_train.head()
target_test.head()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import DecisionBoundaryDisplay

k_values = list(range(1, 31))
accuracies = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors = k)
    model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('classifier', knn)])
    model.fit(attr_train, target_train)
    target_pred = model.predict(attr_test)
    accuracies.append(model.score(attr_train, target_train))
plt.plot(k_values, accuracies, marker='o')
plt.title('Accuracy vs. Number of Neighbors (k)')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

In [ ]:
best_index = np.argmax(accuracies)
best_k = k_values[best_index]
knn = KNeighborsClassifier(n_neighbors=best_k)
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('classifier', knn)])
model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)
print("Best K value: ", best_k)
print(classification_report(target_test,target_pred))

In [ ]:
# Compute the confusion matrix
cm = confusion_matrix(target_test,target_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['neutral or dissatisfied', 'satisfied'])
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix - KNN")
plt.grid(False)
plt.show()
